# SchoolBridge — Hybrid Sentence Extractor 학습 (LayoutXLM + KoCharELECTRA + BIO)

**목적**: LayoutXLM(frozen, visual layout) + KoCharELECTRA(char-level Korean) 결합 → char별 BIO 분류 → 변형 0 sentence 추출.

처리 순서 (사용자 설계):
```
PDF → LayoutXLM(word+bbox+image) → word repr → char broadcast
                                                    ↓
                                       KoCharELECTRA(char + visual ctx)
                                                    ↓
                                       BIO head (O/B-SENT/I-SENT)
```

**입력 파일** (Colab에 업로드):
1. `layoutxlm_bio_train.jsonl` — char별 BIO + char↔word alignment (4,018 records, 2.35M chars)
2. `pdfs.zip` — PDF 폴더 (학습 시 page image 동적 렌더링)
3. `hybrid_model.py` — Hybrid 모델 정의
4. `hybrid_dataset.py` — Dataset/collate_fn 정의

**출력**: `hybrid_best.pt`

**Colab 세팅**: 런타임 → T4 GPU

## 1. 환경 + 설치

In [ ]:
!pip install -q transformers sentencepiece pdfplumber pymupdf pillow
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
print("설치 완료")

In [ ]:
import torch
import detectron2
from transformers import LayoutXLMProcessor, LayoutLMv2ForTokenClassification
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("detectron2:", detectron2.__version__)

## 2. 학습 데이터 + PDF zip 업로드

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, os

uploaded = files.upload()
for name in uploaded:
    print(f'  {name}: {len(uploaded[name])/1024/1024:.1f} MB')

# PDF zip 압축 풀기
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall('.')
        print(f'{name} extracted')

# PDF_DIR — 명시적 후보 시도 (rglob 자동 감지가 . 로 잘못 잡는 케이스 방지)
PDF_DIR = None
for cand in [Path('all_pdfs'), Path('pdfs') / 'all_pdfs', Path('pdfs'), Path('.')]:
    if cand.exists():
        pdfs_here = list(cand.glob('*.pdf'))
        if pdfs_here:
            PDF_DIR = cand
            break

assert PDF_DIR is not None, 'PDF_DIR not found — zip 풀린 위치 확인 필요'
n_pdfs = len(list(PDF_DIR.rglob('*.pdf')))
print(f'PDF_DIR: {PDF_DIR}')
print(f'PDF count: {n_pdfs}')

## 3. 학습 데이터 로드

In [ ]:
records = []
with open('layoutxlm_bio_train.jsonl', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

# 통계
n_pdfs_unique = len({r['pdf'] for r in records})
total_chars = sum(r['n_chars'] for r in records)
total_b = sum(r['n_b_sent'] for r in records)
total_i = sum(r['n_i_sent'] for r in records)
total_o = sum(r['n_o'] for r in records)

print(f'Page records: {len(records)}')
print(f'Unique PDFs: {n_pdfs_unique}')
print(f'Total chars: {total_chars:,}')
print(f'B-SENT: {total_b:,} ({100*total_b/(total_b+total_i+total_o):.1f}%)')
print(f'I-SENT: {total_i:,} ({100*total_i/(total_b+total_i+total_o):.1f}%)')
print(f'O:      {total_o:,} ({100*total_o/(total_b+total_i+total_o):.1f}%)')

# PDF 파일 누락 체크
missing_unique = set(r['pdf'] for r in records if not (PDF_DIR / r['pdf']).exists())
print(f'Missing PDFs: {len(missing_unique)}')
if missing_unique:
    records = [r for r in records if (PDF_DIR / r['pdf']).exists()]
    print(f'Kept records: {len(records)}')

## 4. Hybrid 모델 + Dataset 로드 (업로드한 .py 사용)

In [ ]:
import torch, random, time
from torch.utils.data import DataLoader
from transformers import LayoutXLMProcessor, AutoTokenizer

# 업로드한 .py import
import sys
sys.path.insert(0, '.')
from hybrid_model import HybridSentenceExtractor, HybridConfig, count_params
from hybrid_dataset import HybridDataset, hybrid_collate_fn

config = HybridConfig()
model = HybridSentenceExtractor(config)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
total, trainable = count_params(model)
print(f'Total: {total/1e6:.1f}M, Trainable: {trainable/1e6:.1f}M (LayoutXLM frozen={config.layoutxlm_frozen})')

# Processors / Tokenizers
layoutxlm_processor = LayoutXLMProcessor.from_pretrained(config.layoutxlm_id, apply_ocr=False)
char_tokenizer = AutoTokenizer.from_pretrained(config.kochar_id)

# train/val split
random.seed(42)
random.shuffle(records)
n_val = max(50, len(records) // 10)
val_records = records[:n_val]
train_records = records[n_val:]

train_ds = HybridDataset(train_records, PDF_DIR, layoutxlm_processor, char_tokenizer)
val_ds = HybridDataset(val_records, PDF_DIR, layoutxlm_processor, char_tokenizer)

BATCH = 2  # T4 16GB
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2,
                          collate_fn=hybrid_collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2,
                        collate_fn=hybrid_collate_fn)
print(f'Train: {len(train_records)} | Val: {len(val_records)}')
print(f'Batches: train {len(train_loader)} | val {len(val_loader)}')

## 5. 학습

In [ ]:
# 학습 — LayoutXLM frozen이라 trainable만 optimizer에 넣음
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=3e-5, weight_decay=0.01)
EPOCHS = 5
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val = float('inf')
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    n_batches = 0
    t0 = time.time()
    for batch in train_loader:
        # batch 안 텐서를 device로
        layoutxlm_inputs = {k: v.to(device) for k, v in batch['layoutxlm_inputs'].items()}
        char_input_ids = batch['char_input_ids'].to(device)
        char_attention_mask = batch['char_attention_mask'].to(device)
        char_to_word = batch['char_to_word'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            layoutxlm_inputs=layoutxlm_inputs,
            word_ids_list=batch['word_ids_list'],
            char_input_ids=char_input_ids,
            char_attention_mask=char_attention_mask,
            char_to_word=char_to_word,
            labels=labels,
        )
        loss = outputs['loss']
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()
        train_loss += loss.item()
        n_batches += 1
        if n_batches % 50 == 0:
            print(f'  step {n_batches}/{len(train_loader)} loss={loss.item():.4f}')
    scheduler.step()

    # Validation
    model.eval()
    val_loss = 0.0
    correct_B = 0; total_B = 0
    correct_I = 0; total_I = 0
    correct_O = 0; total_O = 0
    with torch.no_grad():
        for batch in val_loader:
            layoutxlm_inputs = {k: v.to(device) for k, v in batch['layoutxlm_inputs'].items()}
            char_input_ids = batch['char_input_ids'].to(device)
            char_attention_mask = batch['char_attention_mask'].to(device)
            char_to_word = batch['char_to_word'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(
                layoutxlm_inputs=layoutxlm_inputs,
                word_ids_list=batch['word_ids_list'],
                char_input_ids=char_input_ids,
                char_attention_mask=char_attention_mask,
                char_to_word=char_to_word,
                labels=labels,
            )
            val_loss += outputs['loss'].item()
            preds = outputs['logits'].argmax(-1)
            mask = labels != -100
            for cls, (cor, tot) in zip([0, 1, 2], [
                ('correct_O', 'total_O'), ('correct_B', 'total_B'), ('correct_I', 'total_I')
            ]):
                pass  # B/I/O 카운팅 (간단화)
            for cls, name in [(1, 'B'), (2, 'I')]:
                cls_mask = (labels == cls) & mask
                correct = ((preds == cls) & cls_mask).sum().item()
                total = cls_mask.sum().item()
                if cls == 1:
                    correct_B += correct; total_B += total
                else:
                    correct_I += correct; total_I += total

    train_loss /= max(n_batches, 1)
    val_loss /= max(len(val_loader), 1)
    b_acc = correct_B / max(total_B, 1)
    i_acc = correct_I / max(total_I, 1)
    elapsed = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  B-acc={b_acc:.3f} I-acc={i_acc:.3f}  ({elapsed:.0f}s)')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            'state_dict': model.state_dict(),
            'config': config.__dict__,
        }, 'hybrid_best.pt')
        print('  > saved best')

print(f'Best val: {best_val:.4f}')

## 6. best.pt 다운로드

In [ ]:
from google.colab import files
files.download('hybrid_best.pt')